In [5]:
# Healthcare RAG Project

print("Healthcare RAG Project Started!")

project_title = "A Retrieval-Augmented Generation System for Reliable Healthcare Question Answering"

print(project_title)

Healthcare RAG Project Started!
A Retrieval-Augmented Generation System for Reliable Healthcare Question Answering


In [6]:
# Step 2: Create a small healthcare knowledge base

documents = [
    {
        "title": "Hypertension",
        "text": "Hypertension, or high blood pressure, is a condition in which the force of blood against artery walls is consistently too high. Regular blood pressure monitoring, a healthy diet, physical activity, and appropriate medical care can help manage hypertension."
    },
    {
        "title": "Diabetes",
        "text": "Diabetes is a chronic condition that affects how the body regulates blood glucose. Management may include healthy eating, physical activity, monitoring blood glucose, and medicines prescribed by a healthcare professional."
    },
    {
        "title": "Asthma",
        "text": "Asthma is a chronic respiratory condition that can cause episodes of wheezing, coughing, chest tightness, and difficulty breathing. Triggers vary between individuals, and treatment should be guided by a healthcare professional."
    },
    {
        "title": "Anemia",
        "text": "Anemia occurs when the blood does not have enough healthy red blood cells or hemoglobin to carry oxygen effectively. Causes can include iron deficiency and other medical conditions. Evaluation by a healthcare professional is important."
    },
    {
        "title": "Healthy Lifestyle",
        "text": "A healthy lifestyle can include regular physical activity, a balanced diet, adequate sleep, stress management, avoiding tobacco, and limiting harmful alcohol use. Individual health needs can vary."
    }
]

print("Number of documents:", len(documents))

for doc in documents:
    print("\nTitle:", doc["title"])
    print("Text:", doc["text"])

Number of documents: 5

Title: Hypertension
Text: Hypertension, or high blood pressure, is a condition in which the force of blood against artery walls is consistently too high. Regular blood pressure monitoring, a healthy diet, physical activity, and appropriate medical care can help manage hypertension.

Title: Diabetes
Text: Diabetes is a chronic condition that affects how the body regulates blood glucose. Management may include healthy eating, physical activity, monitoring blood glucose, and medicines prescribed by a healthcare professional.

Title: Asthma
Text: Asthma is a chronic respiratory condition that can cause episodes of wheezing, coughing, chest tightness, and difficulty breathing. Triggers vary between individuals, and treatment should be guided by a healthcare professional.

Title: Anemia
Text: Anemia occurs when the blood does not have enough healthy red blood cells or hemoglobin to carry oxygen effectively. Causes can include iron deficiency and other medical conditio

In [7]:
# Step 3: Text Chunking

def create_chunks(text, chunk_size=40):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks


# Create chunks for every healthcare document
all_chunks = []

for doc in documents:
    chunks = create_chunks(doc["text"], chunk_size=40)

    for chunk in chunks:
        all_chunks.append({
            "title": doc["title"],
            "text": chunk
        })


print("Total chunks:", len(all_chunks))

for i, chunk in enumerate(all_chunks):
    print("\nChunk", i + 1)
    print("Topic:", chunk["title"])
    print("Text:", chunk["text"])

Total chunks: 5

Chunk 1
Topic: Hypertension
Text: Hypertension, or high blood pressure, is a condition in which the force of blood against artery walls is consistently too high. Regular blood pressure monitoring, a healthy diet, physical activity, and appropriate medical care can help manage hypertension.

Chunk 2
Topic: Diabetes
Text: Diabetes is a chronic condition that affects how the body regulates blood glucose. Management may include healthy eating, physical activity, monitoring blood glucose, and medicines prescribed by a healthcare professional.

Chunk 3
Topic: Asthma
Text: Asthma is a chronic respiratory condition that can cause episodes of wheezing, coughing, chest tightness, and difficulty breathing. Triggers vary between individuals, and treatment should be guided by a healthcare professional.

Chunk 4
Topic: Anemia
Text: Anemia occurs when the blood does not have enough healthy red blood cells or hemoglobin to carry oxygen effectively. Causes can include iron deficiency a

In [8]:
!pip uninstall -y sentence-transformers transformers torch torchvision torchaudio
!pip install -q torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0
!pip install -q sentence-transformers

Found existing installation: sentence-transformers 6.1.0
Uninstalling sentence-transformers-6.1.0:
  Successfully uninstalled sentence-transformers-6.1.0
Found existing installation: transformers 5.18.0
Uninstalling transformers-5.18.0:
  Successfully uninstalled transformers-5.18.0
Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0
Found existing installation: torchvision 0.26.0
Uninstalling torchvision-0.26.0:
  Successfully uninstalled torchvision-0.26.0
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.21.0 requires transformers, which is not installed.


In [9]:
import torch

print("Torch:", torch.__version__)

Torch: 2.11.0+cu130


In [10]:
from sentence_transformers import SentenceTransformer

print("Sentence Transformers is ready!")

Sentence Transformers is ready!


In [11]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

chunk_texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True
)

print("Number of chunks:", len(chunk_texts))
print("Embedding shape:", embeddings.shape)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!
Number of chunks: 5
Embedding shape: (5, 384)


In [12]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True
)

print("Number of chunks:", len(chunk_texts))
print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of chunks: 5
Embedding shape: (5, 384)


In [13]:
import numpy as np

def semantic_search(query, top_k=2):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )[0]

    similarities = np.dot(embeddings, query_embedding)

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "title": all_chunks[index]["title"],
            "text": all_chunks[index]["text"],
            "score": similarities[index]
        })

    return results


query = "What is high blood pressure?"
results = semantic_search(query, top_k=2)

print("Question:", query)

for i, result in enumerate(results):
    print("\nResult", i + 1)
    print("Topic:", result["title"])
    print("Similarity score:", round(float(result["score"]), 4))
    print("Text:", result["text"])

Question: What is high blood pressure?

Result 1
Topic: Hypertension
Similarity score: 0.7215
Text: Hypertension, or high blood pressure, is a condition in which the force of blood against artery walls is consistently too high. Regular blood pressure monitoring, a healthy diet, physical activity, and appropriate medical care can help manage hypertension.

Result 2
Topic: Anemia
Similarity score: 0.2667
Text: Anemia occurs when the blood does not have enough healthy red blood cells or hemoglobin to carry oxygen effectively. Causes can include iron deficiency and other medical conditions. Evaluation by a healthcare professional is important.


In [14]:
def rag_pipeline(query, top_k=2):
    results = semantic_search(query, top_k=top_k)

    context = "\n\n".join(
        [f"{result['title']}: {result['text']}" for result in results]
    )

    print("QUESTION:")
    print(query)

    print("\n" + "=" * 60)
    print("RETRIEVED CONTEXT")
    print("=" * 60)

    print(context)

    return context


query = "What is hypertension and how can it be managed?"

context = rag_pipeline(query)

QUESTION:
What is hypertension and how can it be managed?

RETRIEVED CONTEXT
Hypertension: Hypertension, or high blood pressure, is a condition in which the force of blood against artery walls is consistently too high. Regular blood pressure monitoring, a healthy diet, physical activity, and appropriate medical care can help manage hypertension.

Diabetes: Diabetes is a chronic condition that affects how the body regulates blood glucose. Management may include healthy eating, physical activity, monitoring blood glucose, and medicines prescribed by a healthcare professional.


In [15]:
try:
    import transformers
    print("Transformers version:", transformers.__version__)
    print("Transformers is ready!")
except Exception as e:
    print("Transformers problem:")
    print(e)

Transformers version: 5.18.0
Transformers is ready!


In [16]:
import torch

print("PyTorch version:", torch.__version__)
print("PyTorch is working!")

PyTorch version: 2.11.0+cu130
PyTorch is working!


In [17]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("LLM loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded successfully!


In [18]:
def generate_answer(question, context):
    prompt = f"""
Answer the healthcare question using ONLY the information provided in the context.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


question = "What is hypertension and how can it be managed?"

answer = generate_answer(question, context)

print("Question:", question)
print("\nRAG Answer:")
print(answer)

Question: What is hypertension and how can it be managed?

RAG Answer:
a condition in which the force of blood against artery walls is consistently too high


In [19]:
test_questions = [
    "What is hypertension?",
    "What are the symptoms of asthma?",
    "How is diabetes managed?",
    "What is anemia?",
    "What are some healthy lifestyle habits?"
]

for question in test_questions:
    context = rag_pipeline(question, top_k=2)
    answer = generate_answer(question, context)

    print("\n" + "=" * 70)
    print("QUESTION:", question)
    print("ANSWER:", answer)

QUESTION:
What is hypertension?

RETRIEVED CONTEXT
Hypertension: Hypertension, or high blood pressure, is a condition in which the force of blood against artery walls is consistently too high. Regular blood pressure monitoring, a healthy diet, physical activity, and appropriate medical care can help manage hypertension.

Diabetes: Diabetes is a chronic condition that affects how the body regulates blood glucose. Management may include healthy eating, physical activity, monitoring blood glucose, and medicines prescribed by a healthcare professional.

QUESTION: What is hypertension?
ANSWER: a condition in which the force of blood against artery walls is consistently too high
QUESTION:
What are the symptoms of asthma?

RETRIEVED CONTEXT
Asthma: Asthma is a chronic respiratory condition that can cause episodes of wheezing, coughing, chest tightness, and difficulty breathing. Triggers vary between individuals, and treatment should be guided by a healthcare professional.

Healthy Lifestyle: 

In [20]:
question = "What is hypertension and how can it be managed?"

for k in [1, 2, 3]:
    print("\n" + "=" * 70)
    print("TOP-K =", k)

    context = rag_pipeline(question, top_k=k)
    answer = generate_answer(question, context)

    print("\nGENERATED ANSWER:")
    print(answer)


TOP-K = 1
QUESTION:
What is hypertension and how can it be managed?

RETRIEVED CONTEXT
Hypertension: Hypertension, or high blood pressure, is a condition in which the force of blood against artery walls is consistently too high. Regular blood pressure monitoring, a healthy diet, physical activity, and appropriate medical care can help manage hypertension.

GENERATED ANSWER:
a condition in which the force of blood against artery walls is consistently too high

TOP-K = 2
QUESTION:
What is hypertension and how can it be managed?

RETRIEVED CONTEXT
Hypertension: Hypertension, or high blood pressure, is a condition in which the force of blood against artery walls is consistently too high. Regular blood pressure monitoring, a healthy diet, physical activity, and appropriate medical care can help manage hypertension.

Diabetes: Diabetes is a chronic condition that affects how the body regulates blood glucose. Management may include healthy eating, physical activity, monitoring blood glucose,

In [21]:
chunk_sizes = [40, 80, 120]

for size in chunk_sizes:
    chunks = []

    for doc in documents:
        doc_chunks = create_chunks(doc["text"], chunk_size=size)

        for chunk in doc_chunks:
            chunks.append({
                "title": doc["title"],
                "text": chunk
            })

    print("\n" + "=" * 60)
    print("CHUNK SIZE:", size)
    print("NUMBER OF CHUNKS:", len(chunks))


CHUNK SIZE: 40
NUMBER OF CHUNKS: 5

CHUNK SIZE: 80
NUMBER OF CHUNKS: 5

CHUNK SIZE: 120
NUMBER OF CHUNKS: 5


In [22]:
chunk_sizes = [40, 80, 120]

for size in chunk_sizes:
    chunks = []

    for doc in documents:
        doc_chunks = create_chunks(doc["text"], chunk_size=size)

        for chunk in doc_chunks:
            chunks.append({
                "title": doc["title"],
                "text": chunk
            })

    print("\n" + "=" * 60)
    print("CHUNK SIZE:", size)
    print("NUMBER OF CHUNKS:", len(chunks))


CHUNK SIZE: 40
NUMBER OF CHUNKS: 5

CHUNK SIZE: 80
NUMBER OF CHUNKS: 5

CHUNK SIZE: 120
NUMBER OF CHUNKS: 5


In [23]:
from sentence_transformers import SentenceTransformer
import numpy as np

query = "What is hypertension and how can it be managed?"

chunk_sizes = [40, 80, 120]

for size in chunk_sizes:

    # Create chunks
    chunks = []

    for doc in documents:
        doc_chunks = create_chunks(doc["text"], chunk_size=size)

        for chunk in doc_chunks:
            chunks.append({
                "title": doc["title"],
                "text": chunk
            })

    # Create embeddings
    texts = [chunk["text"] for chunk in chunks]

    chunk_embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True
    )

    # Query embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )[0]

    # Similarity
    similarities = np.dot(chunk_embeddings, query_embedding)

    # Best result
    best_index = np.argmax(similarities)

    print("\n" + "=" * 70)
    print("CHUNK SIZE:", size)
    print("NUMBER OF CHUNKS:", len(chunks))
    print("BEST MATCH:", chunks[best_index]["title"])
    print("SIMILARITY SCORE:", round(float(similarities[best_index]), 4))
    print("RETRIEVED TEXT:")
    print(chunks[best_index]["text"])


CHUNK SIZE: 40
NUMBER OF CHUNKS: 5
BEST MATCH: Hypertension
SIMILARITY SCORE: 0.7628
RETRIEVED TEXT:
Hypertension, or high blood pressure, is a condition in which the force of blood against artery walls is consistently too high. Regular blood pressure monitoring, a healthy diet, physical activity, and appropriate medical care can help manage hypertension.

CHUNK SIZE: 80
NUMBER OF CHUNKS: 5
BEST MATCH: Hypertension
SIMILARITY SCORE: 0.7628
RETRIEVED TEXT:
Hypertension, or high blood pressure, is a condition in which the force of blood against artery walls is consistently too high. Regular blood pressure monitoring, a healthy diet, physical activity, and appropriate medical care can help manage hypertension.

CHUNK SIZE: 120
NUMBER OF CHUNKS: 5
BEST MATCH: Hypertension
SIMILARITY SCORE: 0.7628
RETRIEVED TEXT:
Hypertension, or high blood pressure, is a condition in which the force of blood against artery walls is consistently too high. Regular blood pressure monitoring, a healthy diet, 

In [24]:
evaluation_data = [
    {
        "question": "What is high blood pressure?",
        "expected_topic": "Hypertension"
    },
    {
        "question": "How is diabetes managed?",
        "expected_topic": "Diabetes"
    },
    {
        "question": "What are the symptoms of asthma?",
        "expected_topic": "Asthma"
    },
    {
        "question": "What is anemia?",
        "expected_topic": "Anemia"
    },
    {
        "question": "What are healthy lifestyle habits?",
        "expected_topic": "Healthy Lifestyle"
    }
]

correct = 0

print("=" * 70)
print("RETRIEVAL ACCURACY EVALUATION")
print("=" * 70)

for item in evaluation_data:

    results = semantic_search(item["question"], top_k=1)

    predicted_topic = results[0]["title"]

    if predicted_topic == item["expected_topic"]:
        correct += 1
        result = "CORRECT"
    else:
        result = "INCORRECT"

    print("\nQuestion:", item["question"])
    print("Expected:", item["expected_topic"])
    print("Retrieved:", predicted_topic)
    print("Result:", result)

accuracy = correct / len(evaluation_data) * 100

print("\n" + "=" * 70)
print("RETRIEVAL ACCURACY:", round(accuracy, 2), "%")
print("=" * 70)

RETRIEVAL ACCURACY EVALUATION

Question: What is high blood pressure?
Expected: Hypertension
Retrieved: Hypertension
Result: CORRECT

Question: How is diabetes managed?
Expected: Diabetes
Retrieved: Diabetes
Result: CORRECT

Question: What are the symptoms of asthma?
Expected: Asthma
Retrieved: Asthma
Result: CORRECT

Question: What is anemia?
Expected: Anemia
Retrieved: Anemia
Result: CORRECT

Question: What are healthy lifestyle habits?
Expected: Healthy Lifestyle
Retrieved: Healthy Lifestyle
Result: CORRECT

RETRIEVAL ACCURACY: 100.0 %


In [25]:
evaluation_questions = [
    "What is hypertension?",
    "How is diabetes managed?",
    "What are the symptoms of asthma?",
    "What is anemia?",
    "What are healthy lifestyle habits?"
]

print("=" * 70)
print("RAG ANSWER EVALUATION")
print("=" * 70)

for question in evaluation_questions:

    context = rag_pipeline(question, top_k=2)
    answer = generate_answer(question, context)

    print("\n" + "-" * 70)
    print("QUESTION:", question)
    print("\nANSWER:")
    print(answer)

RAG ANSWER EVALUATION
QUESTION:
What is hypertension?

RETRIEVED CONTEXT
Hypertension: Hypertension, or high blood pressure, is a condition in which the force of blood against artery walls is consistently too high. Regular blood pressure monitoring, a healthy diet, physical activity, and appropriate medical care can help manage hypertension.

Diabetes: Diabetes is a chronic condition that affects how the body regulates blood glucose. Management may include healthy eating, physical activity, monitoring blood glucose, and medicines prescribed by a healthcare professional.

----------------------------------------------------------------------
QUESTION: What is hypertension?

ANSWER:
a condition in which the force of blood against artery walls is consistently too high
QUESTION:
How is diabetes managed?

RETRIEVED CONTEXT
Diabetes: Diabetes is a chronic condition that affects how the body regulates blood glucose. Management may include healthy eating, physical activity, monitoring blood gl

In [26]:
import pandas as pd

results_data = []

for item in evaluation_data:

    question = item["question"]
    expected_topic = item["expected_topic"]

    retrieved = semantic_search(question, top_k=1)

    predicted_topic = retrieved[0]["title"]
    similarity_score = float(retrieved[0]["score"])

    context = retrieved[0]["text"]

    answer = generate_answer(question, context)

    results_data.append({
        "Question": question,
        "Expected Topic": expected_topic,
        "Retrieved Topic": predicted_topic,
        "Similarity Score": round(similarity_score, 4),
        "Answer": answer
    })

results_df = pd.DataFrame(results_data)

print("RAG Evaluation Results:")
display(results_df)

RAG Evaluation Results:


,Question,Expected Topic,Retrieved Topic,Similarity Score,Answer
0,What is high blood pressure?,Hypertension,Hypertension,0.7215,a condition in which the force of blood agains...
1,How is diabetes managed?,Diabetes,Diabetes,0.7731,Manages
2,What are the symptoms of asthma?,Asthma,Asthma,0.6243,"episodes of wheezing, coughing, chest tightnes..."
3,What is anemia?,Anemia,Anemia,0.7732,a condition that causes iron deficiency
4,What are healthy lifestyle habits?,Healthy Lifestyle,Healthy Lifestyle,0.7663,"regular physical activity, a balanced diet, ad..."


In [27]:
def rag_with_source(question, top_k=2):

    results = semantic_search(question, top_k=top_k)

    context = "\n\n".join(
        [f"{r['title']}: {r['text']}" for r in results]
    )

    answer = generate_answer(question, context)

    print("=" * 70)
    print("QUESTION")
    print(question)

    print("\nANSWER")
    print(answer)

    print("\nSOURCES")
    for i, result in enumerate(results):
        print(
            f"{i+1}. {result['title']} "
            f"(Similarity: {result['score']:.4f})"
        )

        print("   ", result["text"])


question = "What is hypertension and how can it be managed?"

rag_with_source(question, top_k=2)

QUESTION
What is hypertension and how can it be managed?

ANSWER
a condition in which the force of blood against artery walls is consistently too high

SOURCES
1. Hypertension (Similarity: 0.7628)
    Hypertension, or high blood pressure, is a condition in which the force of blood against artery walls is consistently too high. Regular blood pressure monitoring, a healthy diet, physical activity, and appropriate medical care can help manage hypertension.
2. Diabetes (Similarity: 0.3544)
    Diabetes is a chronic condition that affects how the body regulates blood glucose. Management may include healthy eating, physical activity, monitoring blood glucose, and medicines prescribed by a healthcare professional.


In [28]:
import re

def grounding_check(answer, context):
    answer_words = set(
        re.findall(r"\b[a-zA-Z]{4,}\b", answer.lower())
    )

    context_words = set(
        re.findall(r"\b[a-zA-Z]{4,}\b", context.lower())
    )

    if len(answer_words) == 0:
        return 0

    supported_words = answer_words.intersection(context_words)

    grounding_score = len(supported_words) / len(answer_words)

    return grounding_score


question = "What is hypertension and how can it be managed?"

results = semantic_search(question, top_k=2)

context = "\n\n".join(
    [f"{r['title']}: {r['text']}" for r in results]
)

answer = generate_answer(question, context)

score = grounding_check(answer, context)

print("Question:", question)
print("\nAnswer:")
print(answer)

print("\nGrounding Score:",
      round(score * 100, 2), "%")

Question: What is hypertension and how can it be managed?

Answer:
a condition in which the force of blood against artery walls is consistently too high

Grounding Score: 100.0 %


In [29]:
from sentence_transformers import SentenceTransformer
import numpy as np

models = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2"
]

question = "What is hypertension and how can it be managed?"

for model_name in models:

    print("\n" + "=" * 70)
    print("EMBEDDING MODEL:", model_name)

    test_model = SentenceTransformer(model_name)

    test_texts = [chunk["text"] for chunk in all_chunks]

    test_embeddings = test_model.encode(
        test_texts,
        convert_to_numpy=True
    )

    query_embedding = test_model.encode(
        [question],
        convert_to_numpy=True
    )[0]

    similarities = np.dot(
        test_embeddings,
        query_embedding
    )

    best_index = np.argmax(similarities)

    print("Best Topic:", all_chunks[best_index]["title"])
    print(
        "Similarity Score:",
        round(float(similarities[best_index]), 4)
    )
    print("Retrieved Text:")
    print(all_chunks[best_index]["text"])


EMBEDDING MODEL: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Best Topic: Hypertension
Similarity Score: 0.7628
Retrieved Text:
Hypertension, or high blood pressure, is a condition in which the force of blood against artery walls is consistently too high. Regular blood pressure monitoring, a healthy diet, physical activity, and appropriate medical care can help manage hypertension.

EMBEDDING MODEL: all-mpnet-base-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Best Topic: Hypertension
Similarity Score: 0.8234
Retrieved Text:
Hypertension, or high blood pressure, is a condition in which the force of blood against artery walls is consistently too high. Regular blood pressure monitoring, a healthy diet, physical activity, and appropriate medical care can help manage hypertension.


In [30]:
print("=" * 70)
print("HEALTHCARE RAG — RESEARCH RESULTS SUMMARY")
print("=" * 70)

print("\n1. Retrieval Accuracy")
print("   Result: 100% on the 5-question prototype evaluation set")

print("\n2. Grounding Score")
print("   Result: 100% for the tested example")

print("\n3. Top-K Experiment")
print("   Tested: Top-K = 1, 2, 3")

print("\n4. Chunk Size Experiment")
print("   Tested: 40, 80, 120 words")

print("\n5. Embedding Model Experiment")
print("   Tested:")
print("   - all-MiniLM-L6-v2")
print("   - all-mpnet-base-v2")

print("\n" + "=" * 70)
print("PROJECT STATUS: RESEARCH PROTOTYPE COMPLETED")
print("=" * 70)

HEALTHCARE RAG — RESEARCH RESULTS SUMMARY

1. Retrieval Accuracy
   Result: 100% on the 5-question prototype evaluation set

2. Grounding Score
   Result: 100% for the tested example

3. Top-K Experiment
   Tested: Top-K = 1, 2, 3

4. Chunk Size Experiment
   Tested: 40, 80, 120 words

5. Embedding Model Experiment
   Tested:
   - all-MiniLM-L6-v2
   - all-mpnet-base-v2

PROJECT STATUS: RESEARCH PROTOTYPE COMPLETED


In [31]:
evaluation_20 = [
    ("What is hypertension?", "Hypertension"),
    ("What is high blood pressure?", "Hypertension"),
    ("How can hypertension be managed?", "Hypertension"),
    ("Why is blood pressure monitoring important?", "Hypertension"),

    ("What is diabetes?", "Diabetes"),
    ("How is diabetes managed?", "Diabetes"),
    ("What does diabetes affect?", "Diabetes"),
    ("What are some ways to manage blood glucose?", "Diabetes"),

    ("What is asthma?", "Asthma"),
    ("What are common symptoms of asthma?", "Asthma"),
    ("What can asthma cause?", "Asthma"),
    ("Why do asthma triggers vary?", "Asthma"),

    ("What is anemia?", "Anemia"),
    ("What causes anemia?", "Anemia"),
    ("Why is evaluation important for anemia?", "Anemia"),
    ("What does hemoglobin do?", "Anemia"),

    ("What are healthy lifestyle habits?", "Healthy Lifestyle"),
    ("Why is regular physical activity important?", "Healthy Lifestyle"),
    ("What are some healthy lifestyle practices?", "Healthy Lifestyle"),
    ("Why is adequate sleep important?", "Healthy Lifestyle")
]

print("Number of evaluation questions:", len(evaluation_20))

Number of evaluation questions: 20


In [32]:
import numpy as np

top1_correct = 0
top2_correct = 0

print("=" * 70)
print("20-QUESTION RETRIEVAL EVALUATION")
print("=" * 70)

for question, expected_topic in evaluation_20:

    results = semantic_search(question, top_k=2)

    top1_topic = results[0]["title"]
    top2_topics = [r["title"] for r in results]

    if top1_topic == expected_topic:
        top1_correct += 1

    if expected_topic in top2_topics:
        top2_correct += 1

    print("\nQuestion:", question)
    print("Expected:", expected_topic)
    print("Top-1:", top1_topic)
    print("Top-2:", top2_topics)

top1_accuracy = top1_correct / len(evaluation_20) * 100
top2_accuracy = top2_correct / len(evaluation_20) * 100

print("\n" + "=" * 70)
print("FINAL RETRIEVAL RESULTS")
print("=" * 70)

print("Top-1 Accuracy:", round(top1_accuracy, 2), "%")
print("Top-2 Accuracy:", round(top2_accuracy, 2), "%")

20-QUESTION RETRIEVAL EVALUATION

Question: What is hypertension?
Expected: Hypertension
Top-1: Hypertension
Top-2: ['Hypertension', 'Diabetes']

Question: What is high blood pressure?
Expected: Hypertension
Top-1: Hypertension
Top-2: ['Hypertension', 'Anemia']

Question: How can hypertension be managed?
Expected: Hypertension
Top-1: Hypertension
Top-2: ['Hypertension', 'Diabetes']

Question: Why is blood pressure monitoring important?
Expected: Hypertension
Top-1: Hypertension
Top-2: ['Hypertension', 'Diabetes']

Question: What is diabetes?
Expected: Diabetes
Top-1: Diabetes
Top-2: ['Diabetes', 'Hypertension']

Question: How is diabetes managed?
Expected: Diabetes
Top-1: Diabetes
Top-2: ['Diabetes', 'Healthy Lifestyle']

Question: What does diabetes affect?
Expected: Diabetes
Top-1: Diabetes
Top-2: ['Diabetes', 'Anemia']

Question: What are some ways to manage blood glucose?
Expected: Diabetes
Top-1: Diabetes
Top-2: ['Diabetes', 'Healthy Lifestyle']

Question: What is asthma?
Expected